# User Data Stream PoC — 계정 이벤트 (잔고 · 주문 · 포지션)

Market Stream과 달리 **API 키가 필요하다.** 조회 + User Data 권한만 있는 **읽기 전용 키**를
사용하고, 가능하면 먼저 Testnet에서 검증한다.

```bash
export BINANCE_API_KEY='...'
export BINANCE_API_SECRET='...'          # HMAC 키를 쓸 때
export BINANCE_ED25519_KEY_PATH='...'    # Ed25519 개인키 파일 경로 (선택)
```

> **2026-08-26 직접 확인:** Spot의 legacy listenKey 엔드포인트
> (`POST/PUT/DELETE https://api.binance.com/api/v3/userDataStream`)는
> API 키 검증 이전에 CloudFront/nginx 단에서 **`410 Gone`을 반환한다.** 즉 Spot에서 listenKey는
> 이미 완전히 폐지된 상태다. 그래서 이 노트북의 Spot 절은 listenKey 대신 **WebSocket API**
> (`session.logon` / `userDataStream.subscribe.signature`)를 사용한다. USD-M/COIN-M/Options는
> 같은 날짜 기준 listenKey 방식이 정상 동작함을 확인했다(키 없이 호출 시 401).

- 관련 문서: [docs/binanceWebSocketREADME.md](../../binance-api/binanceWebSocketREADME.md)의 5절


In [ ]:
%pip install -q "websockets>=17,<18" "cryptography>=42"

In [6]:
import asyncio
import base64
import hashlib
import hmac
import json
import os
import time
import urllib.request

from websockets.asyncio.client import connect

API_KEY = "rifnyj7vsh0HT5JWVXaS0N7uMTH77NOXKzZv4cd9GHzDG9u9UAd6erTannKZrH8j"
API_SECRET = "vvYZUUABLNq8dtAWe1xB1JQrKC7UszWAEUkOeNqG6PfCtxlTesb6jIS64GaK7Ld1"
ED25519_KEY_PATH = os.environ.get("BINANCE_ED25519_KEY_PATH")

assert API_KEY, "BINANCE_API_KEY 환경변수를 먼저 설정한다 (읽기 전용 키 권장)"

WS_API_URL = "wss://ws-api.binance.com:443/ws-api/v3"

## 1. Spot — WebSocket API 방식 (HMAC 서명)

`session.logon` 없이 요청 하나로 구독하는 방식이다. `BINANCE_API_KEY`/`BINANCE_API_SECRET`만
있으면 된다.

- WS API: `wss://ws-api.binance.com:443/ws-api/v3`
- 메서드: `userDataStream.subscribe.signature`
- 이벤트는 `{"event": {...}}`로 한 겹 감싸져 온다.


In [ ]:
def hmac_signature(params: dict) -> str:
    payload = "&".join(f"{k}={params[k]}" for k in sorted(params))
    return hmac.new(API_SECRET.encode(), payload.encode(), hashlib.sha256).hexdigest()


async def spot_user_data_stream_hmac(message_count: int = 5, timeout: float = 60) -> list[dict]:
    assert API_SECRET, "HMAC 방식은 BINANCE_API_SECRET이 필요하다"

    async with connect(WS_API_URL, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
        params = {"apiKey": API_KEY, "timestamp": int(time.time() * 1000)}
        params["signature"] = hmac_signature(params)
        await ws.send(json.dumps({
            "id": "sub-1",
            "method": "userDataStream.subscribe.signature",
            "params": params,
            })
        )
        print("subscribe response:", json.loads(await ws.recv()))

        events = []
        for _ in range(message_count):
            try:
                raw = await asyncio.wait_for(ws.recv(), timeout=timeout)
            except TimeoutError:
                print(f"{timeout}초 동안 계정 이벤트 없음: {len(events)}건 수신")
                break
            events.append(json.loads(raw))
        return events


spot_events = await spot_user_data_stream_hmac()
spot_events

이벤트를 풀어서 보기 좋게 출력한다. `executionReport`의 `x`(이번에 무슨 일이 일어났나)와
`X`(주문의 현재 상태)를 구분해서 본다.


In [ ]:
for msg in spot_events:
    ev = msg.get("event", msg)
    if ev.get("e") == "executionReport":
        print(ev["s"], "x=" + ev["x"], "X=" + ev["X"], "filled", ev["z"], "@", ev["L"])
    elif ev.get("e") == "outboundAccountPosition":
        print("balance changed:", ev["B"])
    else:
        print(ev)

## 2. Spot — WebSocket API 방식 (Ed25519, `session.logon`)

`BINANCE_ED25519_KEY_PATH`가 설정된 경우에만 실행된다. 한 번 로그인하면 이후 요청에 서명이
필요 없다는 점이 HMAC 방식과 다르다.


In [ ]:
if ED25519_KEY_PATH:
    from cryptography.hazmat.primitives.serialization import load_pem_private_key

    with open(ED25519_KEY_PATH, "rb") as f:
        private_key = load_pem_private_key(f.read(), password=None)

    def ed25519_signature(params: dict) -> str:
        payload = "&".join(f"{k}={params[k]}" for k in sorted(params))
        return base64.b64encode(private_key.sign(payload.encode())).decode()

    async def spot_user_data_stream_ed25519(message_count: int = 5, timeout: float = 60) -> list[dict]:
        async with connect(WS_API_URL, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
            params = {"apiKey": API_KEY, "timestamp": int(time.time() * 1000)}
            params["signature"] = ed25519_signature(params)
            await ws.send(json.dumps({
                "id": "logon-1",
                "method": "session.logon",
                "params": params
                })
            )
            print("logon:", json.loads(await ws.recv()))

            await ws.send(json.dumps({
                "id": "sub-1", 
                "method": "userDataStream.subscribe"
                })
            )
            print("subscribe:", json.loads(await ws.recv()))

            events = []
            for _ in range(message_count):
                try:
                    raw = await asyncio.wait_for(ws.recv(), timeout=timeout)
                except TimeoutError:
                    print(f"{timeout}초 동안 계정 이벤트 없음: {len(events)}건 수신")
                    break
                events.append(json.loads(raw))
            return events

    ed25519_events = await spot_user_data_stream_ed25519()
    ed25519_events
else:
    print("BINANCE_ED25519_KEY_PATH가 설정되지 않아 이 셀은 건너뛴다.")

## 3. USD-M Futures — listenKey 방식

Futures는 여전히 listenKey 방식만 지원한다. 60분 안에 갱신하지 않으면 만료된다.

- REST: `POST/PUT/DELETE https://fapi.binance.com/fapi/v1/listenKey`
- WS: `wss://fstream.binance.com/ws/<listenKey>`


In [ ]:
FUTURES_REST_BASE = "https://fapi.binance.com"


def futures_api_request(method: str, path: str) -> dict:
    req = urllib.request.Request(
        f"{FUTURES_REST_BASE}{path}", method=method, headers={"X-MBX-APIKEY": API_KEY}
    )
    with urllib.request.urlopen(req, timeout=10) as resp:
        return json.load(resp)


listen_key = futures_api_request("POST", "/fapi/v1/listenKey")["listenKey"]
listen_key

실제 운영에서는 60분 이내에 `PUT /fapi/v1/listenKey`로 갱신해야 한다(코드는
[docs/binanceWebSocketREADME.md](../../binance-api/binanceWebSocketREADME.md)의 예제 참고). 이 PoC는
짧게 수신만 확인한다.


In [ ]:
async def futures_user_data_stream(listen_key: str, message_count: int = 5, timeout: float = 60) -> list[dict]:
    url = f"wss://fstream.binance.com/ws/{listen_key}"
    events = []

    async with connect(url, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
        for _ in range(message_count):
            try:
                raw = await asyncio.wait_for(ws.recv(), timeout=timeout)
            except TimeoutError:
                print(f"{timeout}초 동안 계정 이벤트 없음: {len(events)}건 수신")
                break
            events.append(json.loads(raw))

    return events


futures_events = await futures_user_data_stream(listen_key)
futures_events

`ORDER_TRADE_UPDATE`와 `ACCOUNT_UPDATE`를 구분해서 출력한다. Spot의 `executionReport`와 달리
`ps`(positionSide), `rp`(realizedProfit) 필드가 있다.


In [ ]:
for ev in futures_events:
    if ev.get("e") == "ORDER_TRADE_UPDATE":
        o = ev["o"]
        print(o["s"], "x=" + o["x"], "X=" + o["X"], "positionSide", o["ps"], "realizedPnl", o["rp"])
    elif ev.get("e") == "ACCOUNT_UPDATE":
        print("balances", ev["a"]["B"], "positions", ev["a"]["P"])
    else:
        print(ev)

확인이 끝나면 listenKey를 명시적으로 닫아 불필요한 세션을 남기지 않는다.


In [ ]:
futures_api_request("DELETE", "/fapi/v1/listenKey")

## 정리

| 시장 | 연결 방식 | 비고 |
| --- | --- | --- |
| Spot | WebSocket API (`session.logon` 또는 `subscribe.signature`) | listenKey는 `410 Gone`으로 폐지됨 |
| USD-M Futures | listenKey | 60분 만료, 주기적 `PUT` 갱신 필요 |
| COIN-M Futures | listenKey | `https://dapi.binance.com/dapi/v1/listenKey` |
| Options | listenKey | `https://eapi.binance.com/eapi/v1/listenKey` |

운영 구현에서는 Market Stream과 User Data Stream을 **별도 커넥션·별도 큐**로 분리하고,
재연결 후에는 REST(`/api/v3/account`, `/fapi/v2/account` 등)로 상태를 재동기화한다.


## 4. Spot — 요청-응답 조회 (`order.status` / `account.status`)

위의 `userDataStream.subscribe.signature`는 **한 번 구독하면 계속 밀어주는(push) 방식**이었다.
반면 `order.status`/`account.status`는 **물어볼 때마다 그 순간의 스냅샷 한 번**만 주는
요청-응답(pull) 방식이다. 계속 최신 상태가 필요하면 반복 호출(polling)해야 하고, 실시간성이
필요하면 위의 구독(subscribe)을 쓰는 게 맞다.

같은 `WS_API_URL`, 같은 서명 방식(`hmac_signature`)을 그대로 재사용하고 `method`만 바뀐다.


In [ ]:
async def ws_api_request(method: str, extra_params: dict | None = None) -> dict:
    params = {"apiKey": API_KEY, "timestamp": int(time.time() * 1000), **(extra_params or {})}
    params["signature"] = hmac_signature(params)

    async with connect(WS_API_URL, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
        await ws.send(json.dumps({
            "id": f"req-{method}",
            "method": method,
            "params": params
            })
        )
        return json.loads(await ws.recv())

### 4-1. 잔고 조회 — `account.status`

추가 파라미터가 필요 없다. 응답에 자산별 잔고, 계정 권한, rate limit 정보가 담겨 온다.


In [8]:
account_status = await ws_api_request("account.status")
account_status

{'id': None,
 'status': 400,
 'error': {'code': -1135,
  'msg': "Invalid 'id' in JSON request; expected an integer, a string matching '^[a-zA-Z0-9-_]{1,36}$', or null."},
 'rateLimits': [{'rateLimitType': 'REQUEST_WEIGHT',
   'interval': 'MINUTE',
   'intervalNum': 1,
   'limit': 6000,
   'count': 4}]}

### 4-2. 주문 조회 — `order.status`

`symbol`과 `orderId`(또는 `origClientOrderId`)가 필요하다. 아래 `ORDER_ID`는 예시이므로
실제로 실행하려면 내 계정에 존재하는 주문 ID로 바꿔야 한다 — 존재하지 않으면 `status`가
200이 아니라 에러(`code`/`msg`)로 응답이 온다.


In [9]:
ORDER_SYMBOL = "BTCUSDT"
ORDER_ID = 12569099453  # TODO: 실제 존재하는 내 주문 ID로 교체

order_status = await ws_api_request("order.status", {"symbol": ORDER_SYMBOL, "orderId": ORDER_ID})
order_status

{'id': None,
 'status': 400,
 'error': {'code': -1135,
  'msg': "Invalid 'id' in JSON request; expected an integer, a string matching '^[a-zA-Z0-9-_]{1,36}$', or null."},
 'rateLimits': [{'rateLimitType': 'REQUEST_WEIGHT',
   'interval': 'MINUTE',
   'intervalNum': 1,
   'limit': 6000,
   'count': 4}]}

## 5. Spot — `session.logon` 후 서명 없이 조회하기

`session.logon`으로 한 번 인증하면, 이후 같은 연결에서 보내는 SIGNED 요청은
`apiKey`/`signature`를 뺄 수 있다. **단, `timestamp`는 SIGNED 요청이면 로그인 후에도
여전히 넣어야 한다** — 서버가 재전송 공격을 막기 위해 요청 시각은 매번 확인하기 때문이다.

```json
{ "id": "balance-1", "method": "account.status", "params": { "timestamp": 1787498072701 } }
```

`BINANCE_ED25519_KEY_PATH`가 설정된 경우에만 실행된다(2번 섹션과 별개로 독립 실행 가능하게
키 로딩을 다시 포함했다).


In [ ]:
if ED25519_KEY_PATH:
    from cryptography.hazmat.primitives.serialization import load_pem_private_key

    with open(ED25519_KEY_PATH, "rb") as f:
        _private_key = load_pem_private_key(f.read(), password=None)

    def _ed25519_signature(params: dict) -> str:
        payload = "&".join(f"{k}={params[k]}" for k in sorted(params))
        return base64.b64encode(_private_key.sign(payload.encode())).decode()

    async def logon_then_account_status() -> dict:
        async with connect(WS_API_URL, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
            # 1. 로그인 — 여기서 딱 한 번만 서명한다
            logon_params = {"apiKey": API_KEY, "timestamp": int(time.time() * 1000)}
            logon_params["signature"] = _ed25519_signature(logon_params)
            await ws.send(json.dumps({"id": "logon-1", "method": "session.logon", "params": logon_params}))
            print("logon:", json.loads(await ws.recv()))

            # 2. 잔고 조회 — apiKey/signature 없이 timestamp만 넣는다
            await ws.send(json.dumps({
                "id": "balance-1",
                "method": "account.status",
                "params": {"timestamp": int(time.time() * 1000)},
            }))
            return json.loads(await ws.recv())

    balance_after_logon = await logon_then_account_status()
    balance_after_logon
else:
    print("BINANCE_ED25519_KEY_PATH가 설정되지 않아 이 셀은 건너뛴다.")